# Gold-Standard Verification

**You know the right answer, did the AI get it?** Bring a question, a known-correct reference answer (your gold standard), and a model with your own key. The tool asks the model, then runs a single-pass check comparing its answer to your gold standard, and tells you whether it matches, partially matches, or does not match, with a short comment. About a minute, free, no signup.

This notebook is **pre-filled with a demonstration default** so your first run shows the check *running* and commenting on a real example. The point is to watch the mechanism work, not to watch it pass.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **This is a basic first-look check, NOT verification you can rely on professionally.** It uses a single AI judge with a stock prompt, no rubric, no consistency check, no audit trail. A single AI judge can be inconsistent and can flip its result on a re-run. If your profession requires you to verify AI output (legal, tax, medical, accounting, and others), **this tool does not discharge that duty.** It is a first look, not a certification.

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O gold_check_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/gold-check/gold_check_core.py
!wget -q -O prices.json        https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/gold-check/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. One provider's key is enough if the answering and checking models are from the same provider.

### On local Jupyter, environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Your question, your gold standard, and a model

The cell below is pre-filled with a demonstration: a question whose correct answer you already know (the gold standard is a specific refund window), run at a model. Run it as-is to watch the check work.

- `QUESTION` -- what you ask the model.
- `REFERENCE` -- the known-correct answer you are checking against (your gold standard).
- `MODEL` -- the model that answers. `JUDGE_MODEL` -- the model that does the checking (can be the same).

Use the provider's own model-ID strings (the tool is model-agnostic). To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

To check **your own**: set `QUESTION`, `REFERENCE`, and `MODEL` to yours.

In [ ]:
MODEL       = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o-mini'   # the model that does the checking (can be the same or a different one)

QUESTION  = 'In how many days must a customer request a refund under our store policy?'
REFERENCE = ('Under our store policy, a customer must request a refund within 14 days of delivery. '
             'Requests after 14 days are not eligible.')

## 4. Run the check

The model answers your question, then a single checking pass compares that answer to your gold standard and reports match / partial / mismatch with a short comment.

**Read the reminder in the output.** This is a single-pass check: a first look, not professional verification. A single AI judge can be inconsistent and can flip on a re-run. It does not certify anything or discharge any professional duty to verify AI output.

In [ ]:
from gold_check_core import gold_check_core, format_result

result = gold_check_core(question=QUESTION, reference=REFERENCE, model=MODEL, judge_model=JUDGE_MODEL)
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). This is a single-pass first look. For verification you can rely on, rubric-based scoring, running the judge several times to confirm it is consistent, an audit trail, and checking against your whole regulation or policy corpus, see [Features](https://llmprover.pysolvr.com/features/?tool=gold-check).